# Commodity MoE — Data Preparation Dashboard

Run **Kernel → Restart & Run All** to refresh all sections.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors

REGIME_LABELS = {
    1: "Inflationary Growth",
    2: "Stagflation / Risk-off",
    3: "Deflationary Growth",
    4: "Recession / Crisis",
    5: "Transitional",
}
REGIME_COLORS = {1: "#e07b39", 2: "#c0392b", 3: "#2980b9", 4: "#7f8c8d", 5: "#27ae60"}
ASSETS = ["oil", "gold", "copper", "spx", "tnote", "usd"]


def load_or_skip(path: str) -> pd.DataFrame | None:
    p = Path(path)
    if not p.exists():
        print(f"  [missing]  {p}  — run pipeline.py first")
        return None
    df = pd.read_parquet(p)
    mb = df.memory_usage(deep=True).sum() / 1e6
    print(
        f"  [ok]  {p.name:<32}  {df.shape[0]:>5,} rows x {df.shape[1]:>2} cols  "
        f"[{df.index[0].date()} -> {df.index[-1].date()}]  {mb:.1f} MB"
    )
    return df


print("Imports OK")

## 1. Data Inventory

In [ ]:
processed  = load_or_skip("data/processed.parquet")
labeled    = load_or_skip("data/labeled.parquet")
targets    = load_or_skip("data/targets.parquet")
macro_pit  = load_or_skip("data/macro_pit.parquet")

# Summary table
rows = []
for name, df in [("processed", processed), ("labeled", labeled),
                  ("targets", targets), ("macro_pit", macro_pit)]:
    if df is not None:
        rows.append({
            "file": f"data/{name}.parquet",
            "rows": df.shape[0],
            "cols": df.shape[1],
            "start": str(df.index[0].date()),
            "end": str(df.index[-1].date()),
            "MB": round(df.memory_usage(deep=True).sum() / 1e6, 2),
        })
    else:
        rows.append({"file": f"data/{name}.parquet", "rows": "—", "cols": "—",
                     "start": "—", "end": "—", "MB": "—"})

display(pd.DataFrame(rows).set_index("file"))

## 2. Processed Data (prices + z-scores)

In [ ]:
if processed is None:
    print("processed.parquet missing — skipping section")
else:
    print("--- HEAD ---")
    display(processed.head(3))
    print("--- TAIL ---")
    display(processed.tail(3))

In [ ]:
if processed is not None:
    print("--- DESCRIPTIVE STATS ---")
    display(processed.describe().T.round(3))

In [ ]:
if processed is not None:
    imp_cols = [c for c in processed.columns if c.endswith("_was_imputed")]
    imp_pct = processed[imp_cols].mean() * 100
    imp_pct.index = [c.replace("_was_imputed", "") for c in imp_cols]

    fig, ax = plt.subplots(figsize=(8, 3))
    bars = ax.bar(imp_pct.index, imp_pct.values, color="steelblue", edgecolor="white")
    ax.bar_label(bars, fmt="%.2f%%", padding=2, fontsize=9)
    ax.set_title("Forward-fill imputation rate per asset (%)", fontsize=12)
    ax.set_ylabel("% of rows imputed")
    ax.set_ylim(0, max(imp_pct.max() * 1.4, 0.5))
    plt.tight_layout()
    plt.show()

In [ ]:
if processed is not None:
    price_cols = ASSETS
    fig, axes = plt.subplots(2, 3, figsize=(14, 6), sharex=True)
    for ax, col in zip(axes.flat, price_cols):
        ax.plot(processed.index, processed[col], linewidth=0.8, color="#2c3e50")
        ax.set_title(col.upper(), fontsize=11)
        ax.xaxis.set_major_locator(mdates.YearLocator(2))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.tick_params(axis="x", rotation=30)
    fig.suptitle("Asset Price History (aligned, forward-filled)", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
if processed is not None:
    zscore_cols = [f"{a}_zscore" for a in ASSETS]
    fig, axes = plt.subplots(2, 3, figsize=(14, 6))
    for ax, col in zip(axes.flat, zscore_cols):
        data = processed[col].dropna()
        ax.hist(data, bins=60, color="#2980b9", edgecolor="white", alpha=0.85)
        for z in [-2, 2]:
            ax.axvline(z, color="red", linestyle="--", linewidth=0.9)
        ax.set_title(col, fontsize=10)
        ax.set_xlabel("z-score")
    fig.suptitle("Rolling 252-day Z-score distributions (red = ±2σ)", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

## 3. Regime Labels

In [ ]:
if labeled is None:
    print("labeled.parquet missing — skipping section")
else:
    vc = labeled["regime"].value_counts().sort_index()
    total = len(labeled)
    dist = pd.DataFrame({
        "regime": vc.index,
        "name": [REGIME_LABELS[r] for r in vc.index],
        "days": vc.values,
        "pct": (vc.values / total * 100).round(1),
    }).set_index("regime")
    display(dist)

In [ ]:
if labeled is not None:
    vc = labeled["regime"].value_counts().sort_index()
    colors = [REGIME_COLORS[r] for r in vc.index]
    labels = [f"{REGIME_LABELS[r]}\n({v:,}d)" for r, v in zip(vc.index, vc.values)]

    fig, ax = plt.subplots(figsize=(7, 7))
    wedges, texts, autotexts = ax.pie(
        vc.values, labels=labels, colors=colors,
        autopct="%1.1f%%", startangle=140,
        textprops={"fontsize": 10},
    )
    ax.set_title("Regime Distribution (20-day rolling TNOTExSPX)", fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
if labeled is not None:
    fig, ax = plt.subplots(figsize=(14, 1.8))
    regime_series = labeled["regime"]
    for regime, color in REGIME_COLORS.items():
        mask = regime_series == regime
        ax.fill_between(
            labeled.index, 0, 1,
            where=mask, color=color, alpha=0.85,
            label=REGIME_LABELS[regime],
        )
    ax.set_yticks([])
    ax.set_xlim(labeled.index[0], labeled.index[-1])
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_title("Regime Timeline", fontsize=12)
    ax.legend(loc="upper left", bbox_to_anchor=(0, -0.3), ncol=3, fontsize=9)
    plt.tight_layout()
    plt.show()

## 4. Residual Targets (6-month AR-detrended returns)

In [ ]:
if targets is None:
    print("targets.parquet missing — skipping section")
else:
    commodity_assets = ["oil", "gold", "copper"]
    nan_rows = []
    for a in commodity_assets:
        nan_rows.append({
            "asset": a,
            "NaN fwd_ret (last 126 rows)": targets[f"{a}_fwd_ret"].isna().sum(),
            "NaN ar_pred (warmup)": targets[f"{a}_ar_pred"].isna().sum(),
            "valid residuals": targets[f"{a}_residual"].notna().sum(),
        })
    display(pd.DataFrame(nan_rows).set_index("asset"))
    print()
    print("--- SAMPLE (5 random rows with no NaN) ---")
    display(targets.dropna().sample(5, random_state=42))

In [ ]:
if targets is not None:
    commodity_assets = ["oil", "gold", "copper"]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, a in zip(axes, commodity_assets):
        col = f"{a}_fwd_ret"
        data = targets[col].dropna()
        ax.hist(data, bins=60, color="#e07b39", edgecolor="white", alpha=0.85)
        ax.axvline(data.mean(), color="red", linestyle="--", linewidth=1.2, label=f"mean={data.mean():.3f}")
        ax.set_title(f"{a} 6m fwd return", fontsize=11)
        ax.set_xlabel("return")
        ax.legend(fontsize=9)
    fig.suptitle("126-day Forward Return Distributions", fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
if targets is not None:
    commodity_assets = ["oil", "gold", "copper"]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, a in zip(axes, commodity_assets):
        col = f"{a}_residual"
        data = targets[col].dropna()
        ax.hist(data, bins=60, color="#2980b9", edgecolor="white", alpha=0.85)
        ax.axvline(data.mean(), color="red", linestyle="--", linewidth=1.2, label=f"mean={data.mean():.4f}")
        ax.set_title(f"{a} AR(5) residual", fontsize=11)
        ax.set_xlabel("residual")
        ax.legend(fontsize=9)
    fig.suptitle("AR(5) Residual Target Distributions (ML targets)", fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
if targets is not None:
    commodity_assets = ["oil", "gold", "copper"]
    ac_raw, ac_res = [], []
    for a in commodity_assets:
        raw = targets[f"{a}_fwd_ret"].dropna()
        res = targets[f"{a}_residual"].dropna()
        ac_raw.append(raw.autocorr(lag=1))
        ac_res.append(res.autocorr(lag=1))

    x = np.arange(len(commodity_assets))
    width = 0.35
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(x - width/2, ac_raw, width, label="Raw fwd_ret", color="#e07b39")
    ax.bar(x + width/2, ac_res, width, label="AR(5) residual", color="#2980b9")
    ax.set_xticks(x)
    ax.set_xticklabels([a.upper() for a in commodity_assets])
    ax.set_ylabel("Autocorrelation AC(1)")
    ax.set_title("AC(1): Raw forward return vs AR(5) residual", fontsize=12)
    ax.legend()
    ax.axhline(0, color="black", linewidth=0.8)
    for bar in ax.patches:
        ax.annotate(f"{bar.get_height():.3f}",
                    (bar.get_x() + bar.get_width()/2, bar.get_height()),
                    ha="center", va="bottom" if bar.get_height() >= 0 else "top", fontsize=9)
    plt.tight_layout()
    plt.show()

## 5. Macro PIT Features (FRED vintage)

In [ ]:
if macro_pit is None:
    print("macro_pit.parquet not found.")
    print("To generate it, run:")
    print("  from utils.macro_loader import build_macro_pit")
    print("  import pandas as pd")
    print("  processed = pd.read_parquet('data/processed.parquet')")
    print("  build_macro_pit(processed)")
else:
    pit_series = ["UNRATE", "CPIAUCSL", "FEDFUNDS", "A191RL1Q225SBEA"]
    pit_series = [s for s in pit_series if f"{s}_value" in macro_pit.columns]

    fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=False)
    for ax, s in zip(axes.flat, pit_series):
        col = f"{s}_value"
        ax.plot(macro_pit.index, macro_pit[col], linewidth=0.9, color="#2c3e50")
        ax.set_title(s, fontsize=11)
        ax.xaxis.set_major_locator(mdates.YearLocator(2))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.tick_params(axis="x", rotation=30)
    fig.suptitle("Point-in-Time FRED Values (no lookahead)", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
if macro_pit is not None:
    stale_cols = [c for c in macro_pit.columns if c.endswith("_days_stale")]
    series_ids = [c.replace("_days_stale", "") for c in stale_cols]

    fig, axes = plt.subplots(2, 4, figsize=(16, 6))
    for ax, col, sid in zip(axes.flat, stale_cols, series_ids):
        data = macro_pit[col].dropna()
        ax.hist(data, bins=40, color="#27ae60", edgecolor="white", alpha=0.85)
        ax.set_title(sid, fontsize=9)
        ax.set_xlabel("days stale")
    for ax in axes.flat[len(stale_cols):]:
        ax.set_visible(False)
    fig.suptitle("Days Since Last Publication (staleness per series)", fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
if macro_pit is not None:
    rev_cols = [c for c in macro_pit.columns if c.endswith("_revision")]
    series_ids = [c.replace("_revision", "") for c in rev_cols]
    nonzero_counts = [(macro_pit[c] != 0.0).sum() for c in rev_cols]

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.bar(series_ids, nonzero_counts, color="#c0392b", edgecolor="white")
    ax.bar_label(bars, padding=2)
    ax.set_ylabel("Non-zero days")
    ax.set_title("Release + Revision Days per Series", fontsize=12)
    ax.tick_params(axis="x", rotation=25)
    plt.tight_layout()
    plt.show()

    print("\n--- SAMPLE (5 rows with no NaN) ---")
    display(macro_pit.dropna().sample(5, random_state=42))

## 6. Merged Dataset (all available data)

In [ ]:
parts = []
if processed is not None:
    parts.append(processed)
if labeled is not None:
    parts.append(labeled[["regime"]])
if targets is not None:
    parts.append(targets)
if macro_pit is not None:
    parts.append(macro_pit)

if not parts:
    print("No data available — run pipeline.py first")
    merged = None
else:
    merged = pd.concat(parts, axis=1)
    print(f"Merged shape: {merged.shape}")
    print(f"Date range : {merged.index[0].date()} to {merged.index[-1].date()}")
    print()
    nan_summary = merged.isna().sum()
    nan_summary = nan_summary[nan_summary > 0]
    if len(nan_summary):
        print("Columns with NaN values:")
        display(nan_summary.to_frame("NaN count"))
    else:
        print("No NaN values in merged dataset.")

In [ ]:
if merged is not None:
    print("--- 10 random rows ---")
    display(merged.sample(10, random_state=42))

In [ ]:
if merged is not None:
    fig, ax = plt.subplots(figsize=(14, max(4, len(merged.columns) * 0.22)))
    img = ax.imshow(
        merged.isna().T.values,
        aspect="auto",
        cmap="RdYlGn_r",
        interpolation="none",
        vmin=0, vmax=1,
    )
    ax.set_yticks(range(len(merged.columns)))
    ax.set_yticklabels(merged.columns, fontsize=7)

    # x-axis: year ticks
    years = pd.date_range(merged.index[0], merged.index[-1], freq="YS")
    year_positions = [merged.index.searchsorted(y) for y in years]
    ax.set_xticks(year_positions)
    ax.set_xticklabels([str(y.year) for y in years], fontsize=9)

    ax.set_title(
        "Missing Value Map (green = present, red = NaN)\n"
        "NaN clusters: AR warmup rows at start, fwd_ret at end, macro PIT if not generated",
        fontsize=11,
    )
    plt.tight_layout()
    plt.show()